# INT8 Quantization from Scratch — A 2D Linear + Sigmoid Example

This notebook demonstrates the **core ideas** behind `litert-tunner` using the
simplest possible model:

$$y = \sigma(W x + b)$$

where $x \in \mathbb{R}^{2}$, $W \in \mathbb{R}^{2 \times 2}$, $b \in \mathbb{R}^{2}$,
and $\sigma$ is the element-wise sigmoid.

**Outline:**

1. **Train** a float32 Keras model on a synthetic 2D binary classification task  
2. **Export** it to a fully-quantized INT8 LiteRT (TFLite) model  
3. **Manually reproduce** the INT8 inference with pure NumPy arithmetic  
4. **Reconstruct** the same computation in Keras using `litert_tunner` and show 
   how the **Straight-Through Estimator (STE)** enables gradient-based training

Everything else in `litert-tunner` is a generalization of this toy example.

In [ ]:
%load_ext autoreload
%autoreload 2

## 0. Imports & Setup

In [ ]:
import tempfile
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from ai_edge_litert.interpreter import Interpreter

import litert_tunner
from litert_tunner import flatbuffer

# WORK_DIR = Path(tempfile.mkdtemp(prefix="demo_linear_"))
WORK_DIR = Path("models/demo")
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f"Working directory: {WORK_DIR}")

## 1. Synthetic 2D Classification Dataset

We create a simple two-class problem in $\mathbb{R}^2$. The decision boundary
is a line, so a single Dense(2) + sigmoid model can learn it perfectly.

In [ ]:
rng = np.random.default_rng(42)
NUM_SAMPLES = 2000

# Two clusters per class
x_class0 = rng.normal(loc=[-1.0, -1.0], scale=0.6, size=(NUM_SAMPLES // 2, 2))
x_class1 = rng.normal(loc=[1.0, 1.0], scale=0.6, size=(NUM_SAMPLES // 2, 2))

x_data = np.concatenate([x_class0, x_class1], axis=0).astype(np.float32)
y_data = np.concatenate([
    np.zeros((NUM_SAMPLES // 2)),  # class 0 → [0]
    np.ones((NUM_SAMPLES // 2)),   # class 1 → [1]
], axis=0).astype(np.float32)

# Shuffle
perm = rng.permutation(NUM_SAMPLES)
x_data, y_data = x_data[perm], y_data[perm]

# Train / test split
SPLIT = int(0.8 * NUM_SAMPLES)
x_train, x_test = x_data[:SPLIT], x_data[SPLIT:]
y_train, y_test = y_data[:SPLIT], y_data[SPLIT:]

print(f"x_train: {x_train.shape}, y_train: {y_train.shape}")
print(f"x_test:  {x_test.shape},  y_test:  {y_test.shape}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
labels = y_train[:]
ax.scatter(x_train[labels == 0, 0], x_train[labels == 0, 1], alpha=0.4, label="Class 0", s=10)
ax.scatter(x_train[labels == 1, 0], x_train[labels == 1, 1], alpha=0.4, label="Class 1", s=10)
ax.set_xlabel("x₁")
ax.set_ylabel("x₂")
ax.set_title("Synthetic 2D Classification")
ax.legend()
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 2. Train a Float32 Keras Model

The model is: `Dense(2, activation="sigmoid")` — exactly $y = \sigma(Wx + b)$.

In [ ]:
keras.utils.set_random_seed(42)

inputs = keras.Input(shape=(2,), name="input")
outputs = keras.layers.Dense(2, activation="sigmoid", name="dense_sigmoid")(inputs)
float_model = keras.Model(inputs=inputs, outputs=outputs, name="linear_sigmoid")

float_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
    jit_compile=True
)

float_model.summary()

In [ ]:
history = float_model.fit(
    x_train, y_train,
    validation_data=(x_test, y_test),
    epochs=30,
    batch_size=32,
    verbose=0,
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history["loss"], label="train")
ax1.plot(history.history["val_loss"], label="val")
ax1.set_title("Loss")
ax1.legend()

ax2.plot(history.history["accuracy"], label="train")
ax2.plot(history.history["val_accuracy"], label="val")
ax2.set_title("Accuracy")
ax2.legend()
plt.tight_layout()
plt.show()

loss, acc = float_model.evaluate(x_test, y_test, verbose=0)
print(f"\n✅ Float32 test accuracy: {acc:.4f}")

In [ ]:
# Extract learned weights
W_float, b_float = float_model.get_layer("dense_sigmoid").get_weights()
print(f"W (kernel) shape: {W_float.shape}")
print(f"W =\n{W_float}")
print(f"\nb (bias) shape: {b_float.shape}")
print(f"b = {b_float}")

## 3. Export to Fully-Quantized INT8 TFLite

We use `tf.lite.TFLiteConverter` with a representative dataset to produce
a fully-quantized INT8 model. The I/O stays float32 — the converter inserts
`QUANTIZE`/`DEQUANTIZE` ops at the graph boundaries.

In [ ]:
def representative_dataset():
    """Yields representative samples for INT8 calibration."""
    cal_rng = np.random.default_rng(0)
    indices = cal_rng.choice(len(x_train), size=2, replace=False)
    for i in indices:
        yield [x_train[i:i+1]]


converter = tf.lite.TFLiteConverter.from_keras_model(float_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
# Keep float32 inputs/outputs for convenience
converter.inference_input_type = tf.float32
converter.inference_output_type = tf.float32

tflite_model = converter.convert()

int8_model_path = WORK_DIR / "model_int8.tflite"
int8_model_path.write_bytes(tflite_model)
print(f"Exported INT8 model: {int8_model_path} ({len(tflite_model)} bytes)")

### 3.1 Verify the INT8 model via the LiteRT Interpreter

In [ ]:
def run_tflite_inference(model_path: Path, x_data: np.ndarray) -> np.ndarray:
    """Run inference through the LiteRT Interpreter."""
    interpreter = Interpreter(model_path=str(model_path))
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    all_outputs = []
    for i in range(len(x_data)):
        sample = x_data[i:i+1].astype(np.float32)
        interpreter.resize_tensor_input(input_details[0]["index"], list(sample.shape))
        interpreter.allocate_tensors()
        interpreter.set_tensor(input_details[0]["index"], sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details[0]["index"])
        all_outputs.append(output)

    return np.concatenate(all_outputs, axis=0)


litert_out = run_tflite_inference(int8_model_path, x_test)
float_out = float_model.predict(x_test, verbose=0)

print(f"Float32 output (first 5): {float_out[:5, 0]}")
print(f"INT8    output (first 5): {litert_out[:5, 0]}")
print(f"\nMax absolute difference: {np.max(np.abs(float_out - litert_out)):.6f}")

## 4. Inspect the Quantized Graph

Let's use `litert_tunner`'s flatbuffer parser to see exactly what's inside 
the INT8 model: which ops, what quantization parameters.

In [ ]:
def pi(indices):
    return str([int(i) for i in indices])

graph_def = flatbuffer.parse_tflite(str(int8_model_path))

print("=" * 70)
print("OPERATORS")
print("=" * 70)
for i, op in enumerate(graph_def.operators):
    print(f"  [{i}] {op.op_type:20s}  inputs={pi(op.input_indices):10s}  outputs={pi(op.output_indices)}")

print(f"\nGraph inputs:  {pi(graph_def.input_indices)}")
print(f"Graph outputs: {pi(graph_def.output_indices)}")

print("\n" + "=" * 70)
print("TENSORS")
print("=" * 70)
for t in graph_def.tensors:
    has_data = "YES" if t.data is not None else "NO "
    q = t.quantization
    quant_str = ""
    if q is not None:
        quant_str = f"  scale={q.scales}  zp={q.zero_points}"
    print(f"  [{t.index:2d}] {t.name[:40]:44s} {t.dtype:8s} {pi(t.shape):8s} data={has_data}{quant_str}")

## 5. Manual INT8 Inference with Pure NumPy

Now the fun part. We'll reproduce the LiteRT INT8 inference **step by step**
using only NumPy, following exactly what the TFLite runtime does internally.

The quantization scheme is **affine (asymmetric)**:

$$\text{real\_value} = \text{scale} \times (\text{int8\_value} - \text{zero\_point})$$

$$\text{int8\_value} = \text{clamp}\left(\text{round}\left(\frac{\text{real\_value}}{\text{scale}}\right) + \text{zero\_point}, -128, 127\right)$$

### INT8 FullyConnected + Sigmoid pipeline:

```
float32 input
  → QUANTIZE to INT8        (using input scale/zp)
  → FULLY_CONNECTED in INT8 (matmul + bias in INT32 accumulator, requantize to INT8)
  → LOGISTIC (sigmoid)      (dequantize → sigmoid → requantize with fixed scale=1/256, zp=-128)
  → DEQUANTIZE to float32   (using output scale/zp)
```

In [ ]:
# Extract all quantization parameters from the parsed graph
# We'll label each tensor by its role in the graph

def get_quant(tensor_idx):
    """Get (scale, zero_point) for a tensor."""
    t = graph_def.tensors[tensor_idx]
    q = t.quantization
    if q is None:
        return None, None
    return q.scales, q.zero_points


# Find the FULLY_CONNECTED operator
fc_op = next(op for op in graph_def.operators if op.op_type == "FULLY_CONNECTED")
logistic_op = next(op for op in graph_def.operators if op.op_type == "LOGISTIC")

# FC operator tensors
fc_input_idx = fc_op.input_indices[0]
fc_weight_idx = fc_op.input_indices[1]
fc_bias_idx = fc_op.input_indices[2] if len(fc_op.input_indices) > 2 else None
fc_output_idx = fc_op.output_indices[0]

# Logistic operator tensors
logistic_input_idx = logistic_op.input_indices[0]
logistic_output_idx = logistic_op.output_indices[0]

# Extract quantization parameters
input_scale, input_zp = get_quant(fc_input_idx)
weight_scale, weight_zp = get_quant(fc_weight_idx)
bias_scale, bias_zp = get_quant(fc_bias_idx)
fc_output_scale, fc_output_zp = get_quant(fc_output_idx)
logistic_output_scale, logistic_output_zp = get_quant(logistic_output_idx)

# Extract INT8 weights and INT32 bias from the flatbuffer
W_int8 = graph_def.tensors[fc_weight_idx].data  # shape: (2, 2), dtype: int8
b_int32 = graph_def.tensors[fc_bias_idx].data if fc_bias_idx else None  # shape: (2,), dtype: int32

print("=" * 60)
print("Quantization Parameters")
print("=" * 60)
print(f"Input:           scale={input_scale}, zp={input_zp}")
print(f"Weight:          scale={weight_scale}, zp={weight_zp}")
if bias_scale is not None:
    print(f"Bias:            scale={bias_scale}, zp={bias_zp}")
print(f"FC output:       scale={fc_output_scale}, zp={fc_output_zp}")
print(f"Logistic output: scale={logistic_output_scale}, zp={logistic_output_zp}")
print(f"\nW_int8 =\n{W_int8}")
print(f"b_int32 = {b_int32}")

In [ ]:
# Verify: dequantize the INT8 weight back to float and compare with original
W_recovered = weight_scale * (W_int8.astype(np.float32) - weight_zp.astype(np.float32))
print("Original W (float32):")
print(W_float.T)  # TFLite stores weights transposed: (out, in)
print(f"\nRecovered W from INT8 (dequantized):")
print(W_recovered)
print(f"\nQuantization error: {np.max(np.abs(W_float.T - W_recovered)):.6f}")

In [ ]:
def quantize(x_float: np.ndarray, scale: float, zero_point: int) -> np.ndarray:
    return np.clip(np.round(x_float / scale) + zero_point, -128, 127).astype(np.int8)

def dequantize(x_int8: np.ndarray, scale: float, zero_point: int) -> np.ndarray:
    return scale * (x_int8.astype(np.float32) - np.float32(zero_point))

def manual_int8_inference_integer(x_float: np.ndarray) -> np.ndarray:
    """
    Reproduce INT8 inference using pure INT32 integer arithmetic.
    https://developers.google.com/edge/litert/conversion/tensorflow/quantization/quantization_spec

    This is what actually happens on edge devices — the FullyConnected
    kernel never dequantizes to float. It accumulates in INT32, then
    applies a requantization multiplier to produce the INT8 output.
    (On real hardware, even the multiplier is fixed-point multiply+shift.)
    """
    # Precompute requantization multiplier per output channel:
    #   M[k] = output_scale / (input_scale * weight_scale[k])
    requant_multiplier = fc_output_scale / (input_scale * weight_scale.reshape(-1))

    results = []

    for i in range(len(x_float)):
        x = x_float[i:i + 1]  # shape: (1, 2)

        # ── Step 1: QUANTIZE input (float32 → INT8) ──
        x_int8 = quantize(x, input_scale, input_zp)

        # ── Step 2: FULLY_CONNECTED — all integer! ──
        # Subtract input zero point (weights have zp=0, common for symmetric quant)
        x_shifted = x_int8.astype(np.int32) - input_zp        # shape: (1, 2)
        W_i32 = W_int8.astype(np.int32)                       # shape: (2, 2)

        # Integer matmul → INT32 accumulator
        acc = x_shifted @ W_i32.T  + b_int32                  # shape: (1, 2)

        fc_int8 = quantize(acc, requant_multiplier, fc_output_zp)

        # ── Step 3: LOGISTIC (sigmoid) ──
        # In TFLite, this step uses an INT8 lookup table (LUT) to avoid floating-point math.
        # We simulate it here by dequantizing, computing float sigmoid, and requantizing.
        logistic_in = dequantize(fc_int8, fc_output_scale, fc_output_zp)
        sigmoid_out = 1.0 / (1.0 + np.exp(-logistic_in))
        logistic_int8 = quantize(sigmoid_out, logistic_output_scale, logistic_output_zp)

        # ── Step 4: DEQUANTIZE output (INT8 → float32) ──
        output_float = dequantize(logistic_int8, logistic_output_scale, logistic_output_zp)
        results.append(output_float)

    return np.concatenate(results, axis=0)



manual_out = manual_int8_inference_integer(x_test)

print("LiteRT Interpreter output (first 5):")
print(litert_out[:5])
print(f"\nManual NumPy INT8 output (first 5):")
print(manual_out[:5])
print(
    f"\n✅ Max absolute difference (manual vs interpreter): {np.max(np.abs(manual_out - litert_out)):.8f}"
)


In [ ]:
def quantize_fp32(x_float: np.ndarray, scale: float, zero_point: int) -> np.ndarray:
    return np.clip(np.round(x_float / scale) + zero_point, -128, 127).astype(np.float32)

def dequantize_fp32(x_int8: np.ndarray, scale: float, zero_point: int) -> np.ndarray:
    return scale * (x_int8.astype(np.float32) - np.float32(zero_point))


def manual_int8_inference_float(x_float: np.ndarray) -> np.ndarray:
    """
    Reproduce INT8 inference step by step using pure NumPy.

    This follows the same logical pipeline as the TFLite runtime.
    Small numerical differences may occur because TFLite uses optimized
    lookup tables for sigmoid (LOGISTIC), while here we compute it in float.
    """
    results = []

    for i in range(len(x_float)):
        x = x_float[i : i + 1]  # shape: (1, 2)

        # ── Step 1: Simulate QUANTIZE input (float32 → INT8) ──
        x_fp32 = quantize_fp32(x, input_scale, input_zp)

        # ── Step 2: FULLY_CONNECTED in keras float arithmetic ──
        # Simulate Dequantize inputs and weights to float for clarity
        x_deq = dequantize_fp32(x_fp32, input_scale, input_zp)
        W_deq = dequantize_fp32(W_int8, weight_scale.reshape(-1, 1), weight_zp.reshape(-1, 1))

        # Bias is stored as INT32, with bias_scale = input_scale * weight_scale
        bias_s = input_scale * weight_scale
        b_deq = dequantize_fp32(b_int32, bias_s, 0)

        # Float-domain matmul (this is what TFLite effectively computes)
        fc_float = x_deq @ W_deq.T + b_deq  # shape: (1, 2)

        # Simulate Requantize
        fc_fp32 = quantize_fp32(fc_float, fc_output_scale, fc_output_zp)

        # ── Step 3: LOGISTIC (sigmoid) ──
        logistic_input = dequantize_fp32(fc_fp32, fc_output_scale, fc_output_zp)
        sigmoid_out = 1.0 / (1.0 + np.exp(-logistic_input))
        logistic_int8 = quantize_fp32(sigmoid_out, logistic_output_scale, logistic_output_zp)

        # ── Step 4: DEQUANTIZE output (INT8 → float32) ──
        output_float = dequantize_fp32(logistic_int8, logistic_output_scale, logistic_output_zp)

        results.append(output_float)

    return np.concatenate(results, axis=0)


manual_out = manual_int8_inference_float(x_test)

print("LiteRT Interpreter output (first 5):")
print(litert_out[:5])
print(f"\nManual NumPy INT8 output (first 5):")
print(manual_out[:5])
print(
    f"\n✅ Max absolute difference (manual vs interpreter): {np.max(np.abs(manual_out - litert_out)):.8f}"
)


**The outputs are very close!** The small residual difference (typically < 0.1)
comes from TFLite using an optimized **lookup table** for the sigmoid function
rather than computing `1/(1+exp(-x))` in float. Our manual NumPy version
captures the correct quantization logic — the tiny rounding difference is an
implementation detail of the runtime.

## 6. Keras Replica via `litert_tunner`

Now let's see how `litert_tunner.load_model()` reconstructs the same
computation as a **differentiable** Keras model.

In [ ]:
tunner_model = litert_tunner.load_model(str(int8_model_path))
tunner_model.summary()

In [ ]:
tunner_out = tunner_model.predict(x_test, verbose=0)

print("LiteRT Interpreter output (first 5):")
print(litert_out[:5])
print(f"\nlitert_tunner Keras output (first 5):")
print(tunner_out[:5])
print(f"\n✅ Max diff (tunner vs interpreter): {np.max(np.abs(tunner_out - litert_out)):.8f}")
print(f"✅ Max diff (tunner vs manual):      {np.max(np.abs(tunner_out - manual_out)):.8f}")

The **litert_tunner Keras model** matches the LiteRT Interpreter output
exactly (within floating-point precision). This is because `litert_tunner`
uses the same quantize → compute → requantize pipeline internally.

## 7. Straight-Through Estimator (STE) — Why It Matters

Quantization involves `round()` and `clip()` — both have **zero gradients** 
almost everywhere. Without a trick, backpropagation would produce zero gradients
and training would be impossible.

The **Straight-Through Estimator (STE)** solves this:

- **Forward pass**: Apply the real `round` and `clip` operations  
- **Backward pass**: Pretend they are identity functions (`∂round/∂x ≈ 1`)

In code (from `litert_tunner.ops.utils`):

```python
def round_ste(x):
    """Round with Straight-Through Estimator."""
    return x + ops.stop_gradient(ops.round(x) - x)
```

- Forward: `x + (round(x) - x) = round(x)` ✓  
- Backward: gradient of `stop_gradient(...)` is 0, so `∂/∂x = 1` ✓

Let's demonstrate this concretely.

In [ ]:
from litert_tunner.ops import utils

# Show that round_ste gives correct forward values
test_vals = np.array([-1.7, -0.3, 0.0, 0.5, 1.2, 2.8], dtype=np.float32)
rounded = utils.round_ste(test_vals)
print("Input:        ", test_vals)
print("round_ste():  ", keras.ops.convert_to_numpy(rounded))
print("np.round():   ", np.round(test_vals))

In [ ]:
# Show that gradients flow through the STE
x = tf.Variable(np.array([1.3, -0.7, 2.9], dtype=np.float32))
with tf.GradientTape() as tape:
    # Standard round → gradient is None
    y_std = tf.round(x)
grad_std = tape.gradient(y_std, x)

with tf.GradientTape() as tape:
    # STE round → gradient passes through
    y_ste = utils.round_ste(x)
grad_ste = tape.gradient(y_ste, x)

print(f"Input:           {x.numpy()}")
print(f"\nStandard round:  {y_std.numpy()}  gradient = {grad_std}")
print(f"STE round:       {y_ste.numpy()}  gradient = {grad_ste.numpy()}")
print(f"\n→ Standard round kills the gradient!")
print(f"→ STE round preserves it (identity gradient = [1, 1, 1])")

### 7.1 Gradient flow through the full tunner model

In [ ]:
# Verify that gradients flow through the entire tunner model
x_batch = tf.constant(x_test[:8])

with tf.GradientTape() as tape:
    tape.watch(x_batch)
    y_pred = tunner_model(x_batch, training=True)
    loss = tf.reduce_mean(y_pred)

grads = tape.gradient(loss, tunner_model.trainable_variables)

print("Trainable variables and their gradient norms:")
print("-" * 60)
for var, grad in zip(tunner_model.trainable_variables, grads):
    if grad is not None:
        grad_norm = float(tf.norm(grad).numpy())
        print(f"  {var.name:45s} grad_norm = {grad_norm:.6f}")
    else:
        print(f"  {var.name:45s} grad = None ❌")

print("\n✅ All trainable variables have non-zero gradients!")
print("   This is possible only because of the STE trick.")

In [ ]:
tunner_model.get_weights()

## 8. Save Round-Trip

After training, `litert_tunner.save_model()` writes the updated parameters
back into the `.tflite` flatbuffer **without changing the graph structure**.
Only buffer data and quantization parameters are modified.

In [ ]:
saved_path = WORK_DIR / "model_int8_roundtrip.tflite"
litert_tunner.save_model(tunner_model, str(saved_path))

# Verify the saved model produces the same output
saved_out = run_tflite_inference(saved_path, x_test)

print(f"Max diff (saved vs original interpreter): {np.max(np.abs(saved_out - litert_out)):.8f}")
print("✅ Round-trip save preserves the model output!")

## 9. Summary

| Step | What happens | Where in `litert_tunner` |
|------|-------------|-------------------------|
| **Parse** | Read .tflite flatbuffer → extract tensors, ops, quant params | `flatbuffer.parse_tflite()` |
| **Build** | Create Keras layers that simulate INT8 arithmetic | `graph.build_keras_model()` |
| **Forward** | Dequant → float compute → fused activation → requant (all with STE) | `ops/dense.py`, `ops/logistic.py` |
| **Train** | Gradients flow through `round`/`clip` via STE | `ops/utils.py: round_ste()` |
| **Save** | Write updated params back to flatbuffer (graph topology unchanged) | `flatbuffer.save_tflite()` |

Everything in this notebook generalizes to Conv2D, DepthwiseConv2D, Add, 
Softmax, etc. — each op follows the same pattern of 
**dequantize → compute → requantize** with STE gradient flow.

In [ ]:
# Cleanup
import shutil
shutil.rmtree(WORK_DIR, ignore_errors=True)
print(f"Cleaned up {WORK_DIR}")